# 第 6 课：DQN、TD3、SAC 与 PPO 的区别

**学习目标**：结合公式和关键实现片段识别算法，而不是只背名称。

| 算法 | 动作空间 | On/Off policy | 主要理念 |
|---|---|---|---|
| Q-learning | 离散 | Off-policy | 用表格估计 Q |
| DQN | 离散 | Off-policy | 神经网络近似 Q，经验回放 + 目标网络 |
| PPO | 离散/连续 | On-policy | 限制策略单次更新幅度 |
| TD3 | 连续 | Off-policy | 双 Critic、延迟 Actor 更新、目标动作平滑 |
| SAC | 连续（标准版本） | Off-policy | 最大熵随机策略与双 Critic |

这些是**关键计算片段**，不是完整 DQN/TD3/SAC 训练器。完整训练还需要 replay buffer、软目标更新、探索、神经网络与训练循环。

## 6.1 DQN：Bellman target + target network

$$y=r+\gamma(1-\mathrm{terminated})\max_{a'}Q_{\bar\theta}(s',a'),$$

$$L_Q=\mathbb E\big[(Q_\theta(s,a)-y)^2\big].$$

$\bar\theta$ 是较慢更新的目标网络参数，`stop_gradient` 防止错误地通过 TD 目标反向传播。

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp

reward = jnp.array([1., -0.2, 0.5])
terminated = jnp.array([True, False, False])
q_next_target = jnp.array([[10., 9.], [.4, .6], [1.1, 1.0]])
q_current_chosen = jnp.array([.2, .3, .1])
y_dqn = reward + .99 * (~terminated) * jnp.max(q_next_target, axis=1)
loss = jnp.mean((q_current_chosen - jax.lax.stop_gradient(y_dqn))**2)
print('DQN targets:', np.asarray(y_dqn))
print('Bellman MSE:', float(loss))
assert np.isclose(float(y_dqn[0]), 1.)

## 6.2 TD3：两个 Q 网络取最小值

$$a'=\operatorname{clip}(\mu_{\bar\theta}(s')+\epsilon,a_{\min},a_{\max}),$$

$$y_{TD3}=r+\gamma(1-d)\min_{i=1,2}Q_{\bar\phi_i}(s',a').$$

为什么取最小值？减轻 Q 值过估计。TD3 使用确定性 Actor，且 Actor 的更新频率低于 Critic。

In [ ]:
q1_target = jnp.array([1.2, .5, 1.9])
q2_target = jnp.array([.8, .7, 1.5])
y_td3 = reward + .99*(~terminated)*jnp.minimum(q1_target, q2_target)
print('TD3 target:', np.asarray(y_td3))
assert float(y_td3[1]) < float(reward[1] + .99*q2_target[1])

## 6.3 SAC：鼓励高回报，也鼓励策略具有熵

SAC 的 Actor 最小化（省略状态分布记号）：

$$L_\pi(\theta)=\mathbb E_{s,\epsilon}\big[\alpha\log\pi_\theta(f_\theta(\epsilon;s)|s)-Q_\phi(s,f_\theta(\epsilon;s))\big].$$

重参数化技巧 $a=\mu_\theta(s)+\sigma_\theta(s)\epsilon$ 允许在可微 Q 上对采样动作求导；有界动作一般还需 `tanh` 变换与 log-density 修正。

In [ ]:
# 单步、一维高斯 SAC Actor 的可微性演示（不代表完整 SAC）
key = jax.random.key(5)
epsilon = jax.random.normal(key, (1024,))
states = jnp.ones((1024,))*.8

def sac_actor_loss(theta):
    mu = theta[0]*states
    log_std = jnp.clip(theta[1],-3.,1.)
    action = mu + jnp.exp(log_std)*epsilon
    logprob = -0.5*epsilon**2 - log_std - .5*jnp.log(2*jnp.pi)
    q = -(action+states)**2  # 假想、可微的 Q(s,a)；最佳动作约为 -s
    return jnp.mean(.1*logprob-q)

params = jnp.array([0.,-.7])
print('SAC toy loss:', float(sac_actor_loss(params)))
print('dLoss/dTheta:', np.asarray(jax.grad(sac_actor_loss)(params)))

## 6.4 怎样为你的毕设选算法？

**基线优先**：PPO 便于与 HydroGym 相关工作比较，但在线采样可能昂贵；SAC/TD3 可重复利用已有 transition，在 CFD 计算昂贵时有潜在优势。另一方面，代理环境可以低成本生成新数据，是否还应强调样本利用率，需要实证比较。

必须保持环境、动作范围、物理积分步数、奖励定义、随机种子和 CFD 验证预算一致；不能仅用不同论文的 reward 曲线判断算法优劣。

### 练习

1. 写出 DQN replay buffer 存储结构与 batch sampling。
2. 解释为什么 DQN 不能直接用 `max` 穷举连续吹吸流量。
3. 找出 TD3、SAC 与 PPO 都需要价值估计，却为何训练数据复用能力不同。
4. 阅读 [CleanRL PPO](https://docs.cleanrl.dev/rl-algorithms/ppo/) 和 [SAC 原论文](https://arxiv.org/abs/1801.01290)。